In [1]:
import os
import sys
import psutil
import pyarrow as pa
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
from joblib import Parallel, delayed
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, "..")
from paths import resolve

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
_NCPU = os.cpu_count() - 1 or 1
_TOTAL_MEMORY = psutil.virtual_memory().total
_AVAILABLE_MEMORY = psutil.virtual_memory().available
_MEMORY_PER_WORKER = max(100 * 1024**2, _AVAILABLE_MEMORY // (_NCPU + 1))
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | {_TOTAL_MEMORY / 1024**3:.1f}GB total RAM ({_AVAILABLE_MEMORY / 1024**3:.1f}GB available) | pyarrow {pa.__version__}", flush=True)


Running with 47 CPU cores | 92.3GB total RAM (88.1GB available) | pyarrow 24.0.0


Variables

In [2]:
%run ../0_Config/0_variables.ipynb

Correlation matrix — helper

Reusable MI heatmap that can be invoked after step 2 (raw ranking), step 3 (post-dedup) or step 4 (final selection). `%run 0_correlation_matrix.ipynb` then call `plot_mi_top_features(source=...)`.


In [ ]:
import time as _time

def load_mi_matrix(source: str | None = None):
    """Load the MI matrix (rows=features, cols=horizon h1..hN) for the current
    TARGET / FEATURE_DATASET environment.

    Parameters
    ----------
    source : {"ranked", "unique", "selected", None}
        - "ranked"   : output of step 2 (`feature_data_*.parquet`)
        - "unique"   : output of step 4 (`feature_data_unique_*.parquet`)
        - "selected" : output of step 5 — MI values from the post-dedup matrix
                       masked to features that survived the per-horizon
                       selection in `selected_features_*.parquet`.
        - None       : auto-detect — picks the latest available stage.

    Returns
    -------
    (mi_df, source_used) : (pd.DataFrame, str)
    """
    _stem = Path(os.environ['FEATURE_DATASET']).stem
    _target = os.environ['TARGET']
    base = Path(resolve(f"3_Features_select/Selected_features"))

    paths = {
        "ranked":   base / f"{_target}_feature_data_{_stem}.parquet",
        "unique":   base / f"{_target}_feature_data_unique_{_stem}.parquet",
        "selected": base / f"{_target}_selected_features_{_stem}.parquet",
    }

    if source is None:
        for s in ("selected", "unique", "ranked"):
            if paths[s].exists():
                source = s
                break
        else:
            raise FileNotFoundError(f"No MI parquet found for {_target} / {_stem}")

    if source not in paths:
        raise ValueError(f"source must be one of {list(paths)}; got {source!r}")
    if not paths[source].exists():
        raise FileNotFoundError(f"{source!r} parquet missing: {paths[source]}")

    print(f"Loading [{source}] {paths[source]}", flush=True)
    _t = _time.perf_counter()

    if source in ("ranked", "unique"):
        df = pd.read_parquet(paths[source])
        horizon_cols = sorted(
            [c for c in df.columns if c.startswith("h") and c[1:].isdigit()],
            key=lambda c: int(c[1:]),
        )
        mi_df = df.set_index("feature")[horizon_cols]
    else:
        # "selected" — load boolean mask, multiply per-horizon MI from unique matrix
        mask = pd.read_parquet(paths["selected"])  # index=feature, cols=h1..hN (bool)
        unique_path = paths["unique"]
        if not unique_path.exists():
            raise FileNotFoundError(
                f"'selected' source needs the post-dedup MI matrix at {unique_path}"
            )
        unique_df = pd.read_parquet(unique_path).set_index("feature")
        horizon_cols = [c for c in mask.columns if c.startswith("h") and c[1:].isdigit()]
        horizon_cols = sorted(horizon_cols, key=lambda c: int(c[1:]))
        mi_vals = unique_df.reindex(mask.index)[horizon_cols]
        # NaN-out unselected (feature, horizon) pairs so they drop out of top-N
        mi_df = mi_vals.where(mask[horizon_cols].astype(bool))

    print(
        f"  loaded {source}: shape={mi_df.shape} in {_time.perf_counter() - _t:.1f}s",
        flush=True,
    )
    return mi_df, source


OUTPUT_RESOLUTION = int(os.environ["OUTPUT_RESOLUTION"])


In [ ]:
def mi_matrix_per_horizon(mi_df: pd.DataFrame, top_n: int = 20, *, source: str = ""):
    """One heatmap: rows = each horizon, cols = rank 1..top_n,
    cell annotated with feature name + normalised MI value."""
    _cmap = LinearSegmentedColormap.from_list(
        "mi_vibrant", ["#0a0a1a", "#2e1a6e", "#7b2d8b", "#d63a6e", "#f4845f"], N=256
    )

    horizons = list(mi_df.columns)
    n_h = len(horizons)
    mi_norm = 1 - np.exp(-mi_df)  # normalise to [0,1)

    # For each horizon, pick top-N features by normalised MI (descending).
    values = np.zeros((n_h, top_n), dtype=float)
    labels = np.empty((n_h, top_n), dtype=object)
    for i, h in enumerate(horizons):
        s = mi_norm[h].dropna().sort_values(ascending=False).head(top_n)
        values[i, :len(s)] = s.values
        labels[i, :len(s)] = [f"{name}\n{val:.2f}" for name, val in s.items()]

    rank_cols = [f"#{r+1}" for r in range(top_n)]
    horizon_labels = [f"h{int(h[1:])}  ({int(h[1:]) * OUTPUT_RESOLUTION}min)" for h in horizons]

    data = pd.DataFrame(values, index=horizon_labels, columns=rank_cols)
    annot = pd.DataFrame(labels, index=horizon_labels, columns=rank_cols)

    cell_w, cell_h = 1.6, 0.42
    fig_w = max(12, top_n * cell_w + 3)
    fig_h = max(8, n_h * cell_h + 2)

    with plt.style.context("dark_background"):
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        fig.patch.set_facecolor("#1e1e1e")
        ax.set_facecolor("#1e1e1e")

        sns.heatmap(
            data, ax=ax, cmap=_cmap, vmin=0, vmax=1,
            annot=annot.values, fmt="",
            annot_kws={"size": 9, "color": "white"},
            linewidths=0.2, linecolor="#2e2e2e",
            cbar_kws={"label": "MI  (1 − e⁻ᴹᴵ)", "shrink": 0.6, "pad": 0.01},
        )
        ax.xaxis.set_ticks_position("top")
        ax.xaxis.set_label_position("top")
        ax.tick_params(axis="x", labelsize=10, colors="white", rotation=0)
        ax.tick_params(axis="y", labelsize=10, colors="white", rotation=0)
        ax.set_xlabel("Feature rank", color="white", fontsize=10, labelpad=6)
        ax.set_ylabel("Forecast horizon", color="white", fontsize=10)

        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(colors="white")
        cbar.ax.yaxis.label.set_color("white")

        suffix = f" — {source}" if source else ""
        fig.suptitle(
            f"Top {top_n} features by MI — per horizon ({n_h} horizons × {OUTPUT_RESOLUTION}min){suffix}",
            color="white", fontsize=14, y=0.995,
        )
        plt.tight_layout()
        plt.show()


def plot_mi_top_features(source: str | None = None, top_n: int = 10):
    """Convenience wrapper: load the MI matrix for `source` and draw the heatmap.

    `source` is one of {"ranked", "unique", "selected", None (auto)}.
    """
    mi_df, used = load_mi_matrix(source=source)
    mi_matrix_per_horizon(mi_df, top_n=top_n, source=used)


# Standalone run: auto-detect latest available stage
# if __name__ == "__main__":
#     plot_mi_top_features(top_n=10)
